# Nemotron v1, INT8, sherpa-onnx: full test set

Scores the deployed pipeline of fine-tuned Nemotron v1 on the full OpenSLR 104 test set (2,942 clips, 4.76 h):

- **Model:** the INT8 export in `exports/nemotron-v1-560ms/` (official sherpa-onnx script: dynamic, per-tensor INT8 for the encoder, decoder and joiner).
- **Inference:** sherpa-onnx on CPU, real chunk-by-chunk streaming at 560 ms, greedy RNN-T, language `hi-IN`, 0.66 s of silence at the end (as in the official example).
- **Scoring:** the project `score` (strict WER and T-WER), as for every other model.

This is the real streaming number for v1. The NeMo numbers (WER 0.392) came from whole clips with the attention limited to 560 ms.

**Runtime:** CPU only, no NeMo. More CPU cores make it faster. The notebook prints the core count.

**Resumable:** each output is written to Drive as soon as it is ready. After a disconnect, run all cells again. Clips that are already done are skipped.

## Settings

In [1]:
PROJECT = "/content/drive/MyDrive/projects/asr"
MODEL_DIR = f"{PROJECT}/exports/nemotron-v1-560ms"
PROMPT = "hi-IN"
TAIL_S = 0.66  # silence added at the end of each clip (the 50-clip check showed no gain from more)

TEST_ZIP = f"{PROJECT}/openslr104_test.zip"
MANIFEST = f"{PROJECT}/data/processed/openslr104/test_manifest.jsonl"
AUDIO_DIR = "/content/openslr104/test"
LOCAL_MODEL = "/content/model"
OUT_DIR = f"{PROJECT}/results/nemotron-3.5-asr-streaming-0.6b-finetuned-int8"
HYPS = f"{OUT_DIR}/hyps-560ms.jsonl"
SUMMARY = f"{PROJECT}/results/summary.csv"

## Install

In [2]:
!pip install -q sherpa-onnx soundfile jiwer

## Setup

In [3]:
import csv
import datetime
import json
import os
import shutil
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import sherpa_onnx
import soundfile as sf
from google.colab import drive
from tqdm.auto import tqdm

drive.mount("/content/drive")
sys.path.insert(0, f"{PROJECT}/src")
from hinglish_asr.metrics import score
from hinglish_asr.utils import Normalizer

NUM_THREADS = os.cpu_count()
print(f"sherpa-onnx {sherpa_onnx.__version__}, {NUM_THREADS} CPU cores")

Mounted at /content/drive
sherpa-onnx 2.0.3, 12 CPU cores


## Model and audio

The model files and the clips are copied to the local disk, because Drive is slow to read.

In [4]:
model = Path(LOCAL_MODEL)
model.mkdir(exist_ok=True)
for name in ("encoder.int8.onnx", "decoder.int8.onnx", "joiner.int8.onnx", "tokens.txt"):
    if not (model / name).exists():
        shutil.copy(f"{MODEL_DIR}/{name}", model / name)
print({p.name: f"{p.stat().st_size / 2**20:.1f} MB" for p in sorted(model.iterdir())})

if not Path(AUDIO_DIR).exists():
    local_zip = Path("/content/openslr104_test.zip")
    shutil.copy(TEST_ZIP, local_zip)
    with zipfile.ZipFile(local_zip) as z:
        for member in tqdm(z.infolist(), desc="unzip test"):
            z.extract(member, AUDIO_DIR)
    local_zip.unlink()

{'decoder.int8.onnx': '14.3 MB', 'encoder.int8.onnx': '627.1 MB', 'joiner.int8.onnx': '9.1 MB', 'tokens.txt': '0.1 MB'}


unzip test:   0%|          | 0/5886 [00:00<?, ?it/s]

## Manifest

Each clip is found by file name, because the manifest has laptop paths.

In [5]:
wavs = {p.name: str(p) for p in Path(AUDIO_DIR).rglob("*.wav")}
with open(MANIFEST, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f]
missing = [r["utt_id"] for r in rows if Path(r["audio_path"]).name not in wavs]
assert not missing, f"{len(missing)} clips missing, e.g. {missing[:3]}"
paths = {r["utt_id"]: wavs[Path(r["audio_path"]).name] for r in rows}
audio_hours = sum(r["duration"] for r in rows) / 3600
print(f"{len(rows)} clips, {audio_hours:.2f} h")

norm = Normalizer(f"{PROJECT}/data/lists/loanwords.csv", f"{PROJECT}/data/lists/transliterations.csv")
refs = [r["text"] for r in rows]
assert score(refs, refs, norm) == {"wer": 0.0, "t_wer": 0.0}

2942 clips, 4.76 h


## Recognizer

The same settings as the export check. sherpa-onnx reads the feature size and the prompt dictionary from the encoder metadata.

In [6]:
recognizer = sherpa_onnx.OnlineRecognizer.from_transducer(
    tokens=str(model / "tokens.txt"),
    encoder=str(model / "encoder.int8.onnx"),
    decoder=str(model / "decoder.int8.onnx"),
    joiner=str(model / "joiner.int8.onnx"),
    num_threads=NUM_THREADS,
    decoding_method="greedy_search",
)


def decode(wav_path):
    samples, sample_rate = sf.read(wav_path, dtype="float32")
    stream = recognizer.create_stream()
    stream.set_option("language", PROMPT)
    stream.accept_waveform(sample_rate, samples)
    stream.accept_waveform(sample_rate, np.zeros(int(TAIL_S * sample_rate), dtype=np.float32))
    stream.input_finished()
    while recognizer.is_ready(stream):
        recognizer.decode_stream(stream)
    return recognizer.get_result(stream).strip()


for r in rows[:3]:
    print("REF:", r["text"])
    print("HYP:", decode(paths[r["utt_id"]]), "\n")

REF: libre office impress पर slide master और slide design के इस spoken tutorial में आपका स्वागत है
HYP: libreoffice impress पर slide master और slide इस spoken tutorial में आपका स्वागत है 

REF: इस tutorial में हम यह सीखेंगे कि slides के लिए backgrounds layout कैसे लागू करें
HYP: इस tutorial में हम सीखेंगे कि slides के लिए backgrounds layout कैसे लागू करें यहाँ 

REF: background slide पर लागू किये गये सभी रंगों और प्रभावों को उद्घृत करता है जो विषयवस्तु के पीछे मौजूद हैं
HYP: background slide पर लागू किये गए सभी रंगों और प्रभावों को उद्घित करता है जो विषय वस्तु के पीछे मौजूद है 



## Transcribe the test set

Outputs go to `hyps-560ms.jsonl` on Drive, one line for each clip, as soon as each clip is done. On a new run, clips that are already in the file are skipped. A line that a disconnect cut off is removed first.

In [7]:
BATCH = 32

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
done = {}
if Path(HYPS).exists():
    with open(HYPS, encoding="utf-8") as f:
        for line in f:
            try:
                d = json.loads(line)
                done[d["utt_id"]] = d
            except json.JSONDecodeError:
                pass  # a line cut off by a disconnect
    with open(HYPS, "w", encoding="utf-8") as f:  # keep only complete lines
        for d in done.values():
            f.write(json.dumps(d, ensure_ascii=False) + "\n")
print(f"{len(done)} clips already done")

todo = [r for r in rows if r["utt_id"] not in done]
todo_audio_s = sum(r["duration"] for r in todo)
start = time.time()
with open(HYPS, "a", encoding="utf-8") as f:
    for i in tqdm(range(0, len(todo), BATCH), desc=f"transcribe ({BATCH} clips per step)"):
        batch = todo[i:i + BATCH]
        streams = []
        for r in batch:
            samples, sample_rate = sf.read(paths[r["utt_id"]], dtype="float32")
            stream = recognizer.create_stream()
            stream.set_option("language", PROMPT)
            stream.accept_waveform(sample_rate, samples)
            stream.accept_waveform(sample_rate, np.zeros(int(TAIL_S * sample_rate), dtype=np.float32))
            stream.input_finished()
            streams.append(stream)
        while True:
            ready = [s for s in streams if recognizer.is_ready(s)]
            if not ready:
                break
            recognizer.decode_streams(ready)
        for r, stream in zip(batch, streams):
            d = {"utt_id": r["utt_id"], "ref": r["text"], "hyp": recognizer.get_result(stream).strip()}
            f.write(json.dumps(d, ensure_ascii=False) + "\n")
            done[r["utt_id"]] = d
        f.flush()
elapsed = time.time() - start

if todo:
    print(f"{len(todo)} clips in {elapsed / 60:.1f} min, CPU RTF {elapsed / todo_audio_s:.3f} with {NUM_THREADS} cores")

14 clips already done


transcribe (32 clips per step):   0%|          | 0/92 [00:00<?, ?it/s]

2928 clips in 57.0 min, CPU RTF 0.200 with 12 cores


## Score

In [8]:
assert len(done) == len(rows), f"{len(rows) - len(done)} clips missing: run the transcribe cell again"
hyps = [done[r["utt_id"]]["hyp"] for r in rows]
s = score(refs, hyps, norm)
n_empty = sum(not h for h in hyps)

print(f"sherpa-onnx INT8 streaming, 560 ms: WER {s['wer']:.4f}  T-WER {s['t_wer']:.4f}, {n_empty} empty")
print("NeMo v1 (whole clips, attention limited to 560 ms): WER 0.392  T-WER 0.390, 220 empty")

sherpa-onnx INT8 streaming, 560 ms: WER 0.4045  T-WER 0.4020, 236 empty
NeMo v1 (whole clips, attention limited to 560 ms): WER 0.392  T-WER 0.390, 220 empty


## Results file

Adds one row to `results/summary.csv` on Drive, with the same columns as the other rows. If the row is already there, nothing is added.

In [9]:
DECODING = (
    f"fine-tuned v1, sherpa-onnx INT8 (export script default: dynamic, per-tensor), greedy RNN-T, "
    f"real streaming 560 ms chunks, language={PROMPT}, tail {TAIL_S} s, CPU"
)
with open(SUMMARY, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    fields = reader.fieldnames
    exists = any(r.get("decoding") == DECODING for r in reader)

row = {
    "model": "nvidia/nemotron-3.5-asr-streaming-0.6b",
    "dataset": "openslr104",
    "split": "test",
    "n_utts": len(rows),
    "wer": round(s["wer"], 4),
    "t_wer": round(s["t_wer"], 4),
    "decoding": DECODING,
    "date": datetime.date.today().isoformat(),
    "scoring": "v2",
}
unknown = [k for k in row if k not in fields]
assert not unknown, f"summary.csv has no column for: {unknown}"

if exists:
    print("row already in summary.csv, nothing added")
else:
    with open(SUMMARY, "a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=fields).writerow(row)
    print("row added")
print(Path(SUMMARY).read_text(encoding="utf-8").splitlines()[-1])

row added
nvidia/nemotron-3.5-asr-streaming-0.6b,openslr104,test,2942,0.4045,0.402,"fine-tuned v1, sherpa-onnx INT8 (export script default: dynamic, per-tensor), greedy RNN-T, real streaming 560 ms chunks, language=hi-IN, tail 0.66 s, CPU",2026-11-23,v2
